In [1]:
import numpy as np
from pyscf import gto, scf, mp, cc, lo
from pyscf.data import elements

a = 1.20577 # bond length in a cluster
d = 4 # distance between each cluster
unit = 'A' # unit of length
na = 2 # size of a cluster (monomer)
nc = 2 # set as integer multiple of monomers
spin = 0 # spin per monomer
# frozen = 0 # frozen orbital per monomer
elmt = 'N'
basis = 'sto6g'
atoms = ""
for n in range(nc*na):
    shift = ((n - n % na) // na) * (d-a)
    atoms += f"{elmt} {n*a+shift:.5f} 0.00000 0.00000 \n"

mol = gto.M(atom=atoms,
            basis=basis,
            verbose=4,
            unit=unit,
            symmetry=0,
            charge=0,
            spin=spin*nc,
            max_memory=4000,
            )

mf = scf.RHF(mol).density_fit()
mf.kernel()

stable = False
while not stable:
    print(f'mean-field stability test')
    if not stable:
        mo_i, _, stable,_ = mf.stability(return_status=True)
        dm = mf.make_rdm1(mo_i,mf.mo_occ)
        mf.kernel(dm0=dm)
    elif stable:
        print(f'UHF Energy: {mf.e_tot}, stability {stable}')
        break

mymp = mp.MP2(mf).set_frozen()
mymp.kernel()

mycc = cc.CCSD(mf).set_frozen()
mycc.kernel()

print(f"HF   : {mf.e_tot}")
print(f"MP2  : {mymp.e_tot}")
print(f"CCSD : {mycc.e_tot}")

System: uname_result(system='Linux', node='yichi-thinkpad', release='4.4.0-26100-Microsoft', version='#8737-Microsoft Fri Jan 01 08:00:00 PST 2016', machine='x86_64')  Threads 12
Python 3.10.16 | packaged by conda-forge | (main, Dec  5 2024, 14:16:10) [GCC 13.3.0]
numpy 1.24.3  scipy 1.14.1  h5py 3.12.1
Date: Mon Aug  3 19:30:53 2026
PySCF version 2.12.1
PySCF path  /home/yichi/research/software/pyscf
GIT ORIG_HEAD a0665c4a7bf54e33f01295b3eea390be7a17d76d
GIT HEAD (branch master) f97393b29b0a541c155a68d55ee5b652ae7131d2

[ENV] OLD_PYSCF_EXT_PATH /home/sharmagroup/sharmagroup/pyscf-forge:
[ENV] PYSCF_EXT_PATH /home/yichi/research/software/pyscf-forge:/home/sharmagroup/sharmagroup/pyscf-forge:
[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 4
[INPUT] num. electrons = 28
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry 0 subgroup None
[INPUT] Mole.unit = A
[INPUT] Symbol           X                Y                Z      unit          X      

In [2]:
print(f"HF per M   : {mf.e_tot/nc}")
print(f"MP2 per M  : {mymp.e_tot/nc}")
print(f"CCSD per M : {mycc.e_tot/nc}")

HF per M   : -108.53153996327231
MP2 per M  : -108.73152385161768
CCSD per M : -108.71914080642236


In [26]:
frozen = elements.chemcore(mol)
nocc = np.count_nonzero(mf.mo_occ)
print(nocc)
lo_occ = lo.PipekMezey(mol, mf.mo_coeff[:,frozen:nocc]).kernel()
lo_vir = lo.PipekMezey(mol, mf.mo_coeff[:,nocc:]).kernel()
print(lo_occ.shape, lo_vir.shape)
lo_coeff = np.hstack((mf.mo_coeff[:,:frozen],lo_occ,lo_vir))
dm0 = mf.make_rdm1(lo_coeff, mf.mo_occ)
mf1 = scf.RHF(mol).density_fit()
mf1.kernel(dm0=dm0)
print(mf.e_tot - mf1.e_tot)

14


******** <class 'pyscf.lo.pipek.PipekMezey'> ********
conv_tol = 1e-06
conv_tol_grad = None
max_cycle = 100
max_stepsize = 0.05
max_iters = 20
kf_interval = 5
kf_trust_region = 5
ah_start_tol = 1000000000.0
ah_start_cycle = 1
ah_level_shift = 0
ah_conv_tol = 1e-12
ah_lindep = 1e-14
ah_max_cycle = 40
ah_trust_region = 3
init_guess = atomic
pop_method = meta_lowdin
Set conv_tol_grad to 0.000316228
macro= 1  f(x)= 6.3414438292103  delta_f= 6.34144  |g|= 1.05781  4 KF 20 Hx
macro= 2  f(x)= 6.4991025060426  delta_f= 0.157659  |g|= 0.0974465  3 KF 9 Hx
macro= 3  f(x)= 6.4999659632007  delta_f= 0.000863457  |g|= 0.0173974  3 KF 10 Hx
macro= 4  f(x)= 6.4999663535086  delta_f= 3.90308e-07  |g|= 0.000954416  2 KF 10 Hx
macro= 5  f(x)= 6.4999665795233  delta_f= 2.26015e-07  |g|= 0.000194473  2 KF 10 Hx
macro X = 5  f(x)= 6.4999665795233  |g|= 0.000194473  10 intor 14 KF 59 Hx


******** <class 'pyscf.lo.pipek.PipekMezey'> ********
conv_tol = 1e-06
conv_tol_grad = None
max_cycle = 100
max_ste

In [27]:
def check_span(mo1, s1e, mo2, thresh = 1e-10):
    '''
    check if mo1 and mo2 span each other
    return (mo1 span mo2),  (mo2 span mo1)
    '''
    olp11 = mo1.T.conj() @ s1e @ mo1
    olp12 = mo1.T.conj() @ s1e @ mo2
    olp22 = mo2.T.conj() @ s1e @ mo2

    span12 = np.abs(olp12.T.conj() @ olp12 - olp22).max() < thresh
    span21 = np.abs(olp12 @ olp12.T.conj() - olp11).max() < thresh
    # span12 = np.abs(olp12.conj().T @ np.linalg.solve(olp11, olp12) - olp22).max() < thresh
    # span21 = np.abs(olp12 @ np.linalg.solve(olp22, olp12.conj().T) - olp11).max() < thresh

    return span12, span21

mo1 = lo_coeff[:,frozen:nocc]
mo2 = mf.mo_coeff[:,frozen:nocc]
s1e = mf.get_ovlp()
print(check_span(mo1, s1e, mo2, thresh = 1e-10))

(True, True)


In [28]:
mf1.mo_coeff = lo_coeff

mymp1 = mp.MP2(mf1).set_frozen()
mymp1.kernel()

mycc1 = cc.CCSD(mf1).set_frozen()
mycc1.kernel()


print(f"HF   : {mf.e_tot:.10f}  {mf1.e_tot:.10f}")
print(f"MP2  : {mymp.e_tot:.10f} {mymp1.e_tot:.10f}")
print(f"CCSD : {mycc.e_tot:.10f}  {mycc1.e_tot:.10f}")


******** <class 'pyscf.mp.dfmp2.DFRMP2'> ********
nocc = 10, nmo = 16
frozen orbitals 4
max_memory 4000 MB (current use 482 MB)
E(DFRMP2) = -217.425640991735  E_corr = -0.362561065190049
E(SCS-DFRMP2) = -217.41469831337  E_corr = -0.351618386825537
E_corr(same-spin) = -0.0962941054644479
E_corr(oppo-spin) = -0.266266959725601

******** <class 'pyscf.cc.dfccsd.RCCSD'> ********
CC2 = 0
CCSD nocc = 10, nmo = 16
frozen orbitals 4
max_cycle = 50
direct = 0
conv_tol = 1e-07
conv_tol_normt = 1e-05
diis_space = 6
diis_start_cycle = 0
diis_start_energy_diff = 1e+09
max_memory 4000 MB (current use 482 MB)
Init t2, MP2 energy = -217.418733093377  E_corr(MP2) -0.35565316683221
Init E_corr(RCCSD) = -0.355653166832223
cycle = 1  E_corr(RCCSD) = -0.338107348950446  dE = 0.0175458179  norm(t1,t2) = 0.0646003
cycle = 2  E_corr(RCCSD) = -0.363532307589759  dE = -0.0254249586  norm(t1,t2) = 0.0385197
cycle = 3  E_corr(RCCSD) = -0.36461490036627  dE = -0.00108259278  norm(t1,t2) = 0.0174646
cycle = 4  E_

In [29]:
print(mf.mo_energy)

[-15.68688817 -15.68688754 -15.68546159 -15.68546154  -1.34865068
  -1.34862686  -0.74506196  -0.74410515  -0.51544817  -0.51381302
  -0.50573506  -0.50573506  -0.50564656  -0.50564656   0.22952482
   0.22952482   0.22972075   0.22972075   0.87417184   0.87567826]


In [33]:
mo = mf.mo_coeff
fock_ao = mf.get_fock()
fock_mo = mo[:,frozen:].T @ fock_ao @ mo[:,frozen:]
fock_lo = lo_coeff[:,frozen:].T @ fock_ao @ lo_coeff[:,frozen:]
print(abs(fock_mo - np.diag(fock_mo.diagonal())).max())
print(fock_mo.diagonal())
print(fock_lo.diagonal())
print(sum(fock_mo.diagonal()[:nocc-frozen]))
print(sum(fock_lo.diagonal()[:nocc-frozen]))

6.311575732821056e-08
[-1.34865067 -1.34862686 -0.74506196 -0.74410515 -0.51544817 -0.51381301
 -0.50573505 -0.50573505 -0.50564656 -0.50564656  0.22952482  0.22952482
  0.22972076  0.22972076  0.87417185  0.87567827]
[-1.12436719 -0.81625699 -0.74009964 -0.74092387 -0.74092739 -0.74011176
 -0.50569097 -0.50569082 -0.81625636 -0.50814403  0.22962283  0.24530104
  0.56057804  0.55201033  0.56022457  0.52060446]
-7.2384690351505
-7.238469035150503


In [34]:
import jax
jax.config.update("jax_enable_x64", True)
import opt_einsum as oe

In [103]:
from afqmc import integral
integral.prep_integral(mycc1, chol_cut=1e-10)


Preparing AFQMC calculation
CCSD type input object
Calculating Cholesky integrals
Find Density Fit Teonsers in MF object
Integrals will be built by DF Tensors
Cholesky shape: (113, 16, 16) 
Finished calculating Cholesky integrals
Size of the correlation space:
Number of electrons:        [10, 10]
Number of basis functions:  16
Number of Cholesky vectors: 113


In [104]:
options =  {'n_blocks': 100,
            'n_walkers': 100,
            'max_memory': 8000,
            'seed': 17,
            'trial': 'pt2ccsd_bar',
            'mix_precision': False,
            }

In [105]:
def get_rfock(nocc, h1, chol):
    jeff = oe.contract('gpq,gjj->pq', chol, chol[:,:nocc,:nocc], backend="jax")
    keff = oe.contract('gpj,gjq->pq', chol[:,:,:nocc], chol[:,:nocc,:], backend="jax")
    fock = h1 + 2 * jeff - keff
    return fock

In [106]:
import time

import numpy as np

from afqmc import config, prep, sampling

from functools import partial

print = partial(print, flush=True)
init_time = time.time()

prep.print_start()
config.setup_jax()

ham_data, ham, prop, trial, wave_data, sampler, options = prep.init_afqmc(options)

wave_data["rdm1"] = trial.get_rdm1(wave_data)
ham_data = ham.build_measurement_intermediates(ham_data, trial, wave_data)
ham_data = ham.build_propagation_intermediates(ham_data, prop, trial, wave_data)
h0 = ham_data['h0']

prop_data = prep.init_hf_prop_data(trial, wave_data, ham_data, options)

init_e = prop_data["e_estimate"]
print(f"AFQMC Init energy : {init_e}")
print(f"PYSCF HF Energy :   {mf.e_tot}")


    ________                     _____                    
    ___  __ \___  __________________(_)_____________ _    
    __  /_/ /  / / /_  __ \_  __ \_  /__  __ \_  __ `/    
    _  _, _// /_/ /_  / / /  / / /  / _  / / /  /_/ /     
    /_/ |_| \__,_/ /_/ /_//_/ /_//_/  /_/ /_/_\__, /      
                                             /____/       
    _____________________________  ___________            
    ___    |__  ____/_  __ \__   |/  /_  ____/            
    __  /| |_  /_   _  / / /_  /|_/ /_  /                 
    _  ___ |  __/   / /_/ /_  /  / / / /___               
    /_/  |_/_/      \___\_\/_/  /_/  \____/               

Hostname:     yichi-thinkpad
System:       Linux
Node:         yichi-thinkpad
Release:      4.4.0-26100-Microsoft
Machine:      x86_64
Processor:    x86_64
JAX backend:  CPU
JAX devices:  [CpuDevice(id=0)]
Device kind:  cpu
Platform:     cpu

QMC Parameters
n_blocks        -        100
n_walkers       -        100
max_memory      -       8000
seed

Maximum memory per walker:            80.00 MB
Maximum number of Cholesky per chunk: 20480
Number of Cholesky chunks:            1
Number of Cholesky per chunk:         113
Number of padding Cholesky:           0

QMC System
Number of electrons: (10, 10)
Spin Multiplicity:   0
Number of orbitals:  16
Number of Chol:      113

Initalize QMC walkers by HF
AFQMC Init energy : -217.06307992647217
PYSCF HF Energy :   -217.06307992654462


In [107]:
from jax import jit, lax
from jax import numpy as jnp

from afqmc import slater_tools

from functools import partial


In [108]:
walker_init = prop_data["walkers"][0]

norb, nocc = walker_init.shape

walker = jnp.array(np.random.rand(*walker_init.shape))

bra = jnp.eye(norb)[:,:nocc]

h0 = ham_data["h0"]
h1 = ham_data["h1"][0]
chol = ham_data["chol"].reshape(-1, norb, norb)

my_fock = get_rfock(nocc, h1, chol)
print(abs(my_fock-fock_mo).max())
print(abs(my_fock-fock_lo).max())

# ene_1 = slater_tools.u_energy(bra, walker, h0, h1, chol)
# print(ene_1)

# ene_2 = slater_tools.u_energy_corr(bra, walker, fock_mo, chol)
# print(ene_2, mf.e_tot + ene_2)

# ene_3 = slater_tools.u_energy_corr(bra, walker, my_fock, chol)
# print(ene_3, mf.e_tot + ene_3)

0.5323698626784165
2.097025331160296e-11


In [109]:
from afqmc.lno_afqmc import lno_afqmc, tools
from pyscf.data import elements
iao_coeff, frag_lolist, atm_center = tools.iao_localization(mf1)

mo1 = lo_occ
mo2 = iao_coeff
s1e = mf.get_ovlp()
print(check_span(mo1, s1e, mo2, thresh = 1e-10))

(False, True)


In [110]:
nfrag = len(frag_lolist)
print(nfrag)
pfrag = [None] * nfrag
print(len(pfrag))
# <lo|mo>
for frag_idx in range(nfrag):
    frag_orb = iao_coeff[:,frag_lolist[frag_idx]]
    frag2mo = frag_orb.T.conj() @ s1e @ lo_occ
    pfrag[frag_idx] = frag2mo.T.conj() @ frag2mo

4
4


In [111]:
def r_energy_corr_frag(bra, ket, fock, chol, pfrag):
    '''
    calculate the correlation energy
    '''
    if len(chol.shape) == 3:
        chol = chol.reshape(1,*chol.shape)

    norb, nocc = ket.shape
    rot_fock = fock[:nocc,nocc:]
    rot_chol = chol[:,:,:nocc,nocc:]

    green = (ket.dot(jnp.linalg.inv(ket[:nocc, :]))).T
    green = green[:nocc,nocc:]
    e1 = oe.contract('ia,ik,ka->', green, pfrag, rot_fock, backend="jax") * 2

    def scan_chol(carry, x):
        chol_c = x  # (nchol_chunk, nocc, nvir)
        lg_c = oe.contract('gia,ja->gij', chol_c, green, backend="jax")
        trlg_c = oe.contract('gik,ik->g', lg_c, pfrag, backend="jax")
        e1_c = oe.contract('g,gii->', trlg_c, lg_c, backend="jax") * 2
        e2_c = oe.contract('gij,gjk,ik->', lg_c, lg_c, pfrag, backend="jax")
        carry += e1_c - e2_c
        return carry, 0.0

    e2, _ = lax.scan(scan_chol, 0.0, rot_chol)
    
    return e1 + e2

In [112]:
walker = jnp.array(np.random.rand(*walker_init.shape) + 1j * np.random.rand(*walker_init.shape)) / 2
print(mf.e_tot + r_energy_corr(walker_init, walker, my_fock, chol))
print(slater_tools.r_energy(walker_init, walker, h0, h1, chol))

(-216.8972645892845-0.27167889251865196j)
(-216.89726458921234-0.2716788925186885j)


In [113]:
ene_4 = 0
for p in pfrag:
    ene_4 += r_energy_corr_frag(bra, walker, my_fock, chol, p)
print(ene_4, mf.e_tot + ene_4)

(0.16581533726012337-0.27167889251869537j) (-216.8972645892845-0.27167889251869537j)


In [114]:
from jax import scipy as jsp
import opt_einsum as oe

t1 = jnp.array(wave_data["t1"])
# t2 = jnp.array(wave_data["t2"])

nocc, nvir = t1.shape
norb = nocc + nvir
t1_full = np.zeros((norb, norb))
t1_full[:nocc, nocc:] = t1
wave_data['exp_t1']  = jsp.linalg.expm(jnp.array(t1_full))
wave_data['exp_mt1'] = jsp.linalg.expm(jnp.array(-t1_full))
# t2 projected in both cases
# wave_data["t2"] = oe.contract('iajb,ik->kajb', t2, prjlo, backend='jax')
ham_data["h1bar"] = wave_data['exp_t1'] @ h1 @ wave_data['exp_mt1']

chol_bar = oe.contract('pr,grs,sq->gpq', wave_data['exp_t1'], chol, wave_data['exp_mt1'], backend='jax')
ham_data["chol_bar"] = chol_bar
fock_bar = get_rfock(nocc, ham_data["h1bar"], ham_data["chol_bar"])

In [115]:
print(mycc.energy(t1=mycc.t1, t2=mycc.t2*0))
print(mf.e_tot + mycc.energy(t1=mycc.t1, t2=mycc.t2*0))

1.9499035175324426e-05
-217.06306042750944


In [117]:
bra_tilde = wave_data['exp_t1'].T @ bra

walker_bar = wave_data['exp_t1'] @ walker

ene5 = slater_tools.r_energy(bra_tilde, bra, h0, h1, chol)
print(ene5)
print(ene5 - mf.e_tot)

ene6 = slater_tools.r_energy(bra, walker_bar, h0, ham_data["h1bar"], ham_data["chol_bar"])
print(ene6)

ecorr_bar = r_energy_corr(bra, walker_bar, fock_bar,  ham_data["chol_bar"])
print(ene5 + ecorr_bar)

-217.06306042746914
1.949907547782459e-05
(-216.95027255967318-0.3007978836964711j)
(-216.95027255967287-0.3007978836964772j)


In [120]:
def ecc_t1(t1, chol):
    lt1 = oe.contract('ia,gja->gij', t1, chol[:, :nocc, nocc:], backend='jax')
    ecc_t1 = 2 * oe.contract('gii,gjj->',lt1, lt1, backend='jax') \
            - oe.contract('gij,gji->', lt1, lt1, backend='jax')
    return ecc_t1

print(ecc_t1(wave_data["t1"], chol) - mycc.energy(t1=mycc.t1, t2=mycc.t2*0))

-6.738558997704627e-11


In [121]:
def ecc_t1_frag(t1, chol, pfrag):
    lt1 = oe.contract('ia,gja->gij', t1, chol[:, :nocc, nocc:], backend='jax')
    ecc_t1 = 2 * oe.contract('gik,ik,gjj->', lt1, pfrag, lt1, backend='jax') \
            - oe.contract('gij,gjk,ik->', lt1, lt1, pfrag, backend='jax')
    return ecc_t1

ene_4 = 0
for p in pfrag:
    ene_4 += ecc_t1_frag(t1, chol, p)
print(ene_4, ene_4 - ecc_t1(wave_data["t1"], chol))

1.9498967789730488e-05 -1.7957098481791167e-19
